In [ ]:
# Step 1: Configure the MLflow server and credentials used to track the analysis. Do not commit real passwords.

MLFLOW_SERVER = "http://10.0.100.20:5000"
MLFLOW_USERNAME = "csc-4933-001-user01"
MLFLOW_PASSWORD = "dG7hrCzKxFyJfP9v"

## 1. Hypothesis and Experiment Design

### Hypothesis 2 - revised for daily data

**Original M1 wording**

If outdoor temperatures decrease, then hourly electricity demand and estimated grid CO2 emissions will increase because colder conditions can increase electricity use for heating and change the electricity generation mix.

**Revised M2 wording**

If daily average outdoor temperatures decrease, then daily average electricity demand and daily estimated grid CO2 emissions will increase because colder conditions can increase electricity use for heating and change the electricity generation mix.

**Reason for revision:** The prepared weather and grid datasets contain daily averages rather than matched hourly observations. The hypothesis was changed to daily measurements so the compared variables use the same time interval. This analysis cannot make claims about specific hourly or nighttime charging patterns.

### Predictions tested separately

1. **Electricity demand:** The coldest daily-temperature quartile will have higher mean daily average electricity demand than the warmest quartile.
2. **Grid emissions:** The coldest daily-temperature quartile will have higher mean estimated daily grid CO2 emissions than the warmest quartile.

The predictions are evaluated independently because higher electricity demand does not necessarily mean higher emissions; the generation mix also matters.

## 2. Load and Inspect the Prepared Daily Data

The input files are:

- `combined_daily_averages.csv`: daily weather, demand, generation, and recorded-hour measurements.
- `fuel_factors_clean.csv`: available fuel emissions factors in kg CO₂/MWh.

The inspection checks the date range, required columns, missing values, duplicate dates, and recorded hours before the modeling dataset is created.

In [ ]:
import os
import getpass
import requests
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)

daily_data = pd.read_csv("combined_daily_averages.csv")
fuel_factors_clean = pd.read_csv("fuel_factors_clean.csv")

daily_data["date"] = pd.to_datetime(daily_data["date"])
fuel_factors_clean["fuel"] = (
    fuel_factors_clean["fuel"].astype(str).str.strip().str.lower()
)

print("Daily dataset shape:", daily_data.shape)
print("Fuel-factor dataset shape:", fuel_factors_clean.shape)
print("Date range:", daily_data["date"].min(), "to", daily_data["date"].max())
print("Duplicate dates:", daily_data["date"].duplicated().sum())

display(daily_data.head())
display(fuel_factors_clean)
display(daily_data.isna().sum().rename("missing_values").to_frame())
display(daily_data["hours_recorded"].value_counts().sort_index())

### 2.1 Inspect daily patterns

These initial visuals show the temperature distribution and the time series of daily average temperature and electricity demand. They are descriptive checks, not hypothesis verdicts.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)

axes[0].plot(daily_data["date"], daily_data["temp_f"], linewidth=0.8)
axes[0].set_title("Daily Average Outdoor Temperature")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Temperature (°F)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(daily_data["date"], daily_data["demand_mwh"], linewidth=0.8)
axes[1].set_title("Daily Average Electricity Demand")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Demand (MWh)")
axes[1].grid(True, alpha=0.3)

plt.show()

plt.figure(figsize=(8, 5))
plt.hist(daily_data["temp_f"].dropna(), bins=25)
plt.title("Distribution of Daily Average Outdoor Temperature")
plt.xlabel("Temperature (°F)")
plt.ylabel("Number of days")
plt.grid(True, alpha=0.3)
plt.show()

## 3. Prepare the Modeling Data and Estimate Daily Grid CO₂

### 3.1 Required measurements

The analysis uses `date`, `temp_f`, `demand_mwh`, generation by source, and `hours_recorded`. Calendar variables are created for model construction.

### 3.2 Generation-unit assumption

**Confirm this against the data dictionary/source before running the emissions calculation.** The code below assumes the generation columns (`coal`, `natural_gas`, etc.) are daily average hourly generation in MWh. Under that assumption, multiplying by `hours_recorded` converts the average hourly value to an estimated daily MWh total. If the columns already contain daily totals, set the configuration variable to `False` so they are not multiplied again.

The `demand_mwh` column is treated as daily average hourly demand and is not multiplied by `hours_recorded` for the demand comparison.

In [ ]:
required_columns = [
    "date", "temp_f", "demand_mwh", "net_generation",
    "coal", "natural_gas", "nuclear", "wind",
    "solar", "hydro", "other", "hours_recorded"
]

missing_columns = [c for c in required_columns if c not in daily_data.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

generation_fuels = [
    "coal", "natural_gas", "nuclear", "wind",
    "solar", "hydro", "other"
]

# Set False only if source documentation says generation columns are already daily totals.
GENERATION_COLUMNS_ARE_DAILY_AVERAGES = True

emission_factors = (
    fuel_factors_clean
    .drop_duplicates(subset=["fuel"], keep="last")
    .set_index("fuel")["kg_co2_per_mwh"]
    .to_dict()
)

# Coal and natural gas are emitting sources and must have explicit factors.
for required_fuel in ["coal", "natural_gas"]:
    if required_fuel not in emission_factors:
        raise ValueError(f"Missing required emissions factor for {required_fuel}.")
    if pd.isna(emission_factors[required_fuel]):
        raise ValueError(f"Emissions factor for {required_fuel} is missing.")

# A zero-emission source may be absent from the factor file because its factor is zero.
# Missing factors for the other listed generation categories are therefore treated as zero.
for fuel in generation_fuels:
    factor = emission_factors.get(fuel, 0.0)
    if pd.isna(factor):
        raise ValueError(f"Emissions factor for {fuel} is missing.")
    emission_factors[fuel] = float(factor)

for fuel in generation_fuels:
    if GENERATION_COLUMNS_ARE_DAILY_AVERAGES:
        daily_data[f"{fuel}_daily_mwh"] = (
            daily_data[fuel] * daily_data["hours_recorded"]
        )
    else:
        daily_data[f"{fuel}_daily_mwh"] = daily_data[fuel]

    daily_data[f"{fuel}_co2_kg"] = (
        daily_data[f"{fuel}_daily_mwh"] * emission_factors[fuel]
    )

emission_columns = [f"{fuel}_co2_kg" for fuel in generation_fuels]
daily_data["daily_co2_kg"] = daily_data[emission_columns].sum(axis=1)

display(
    daily_data[
        ["date", "hours_recorded", "demand_mwh", "daily_co2_kg"]
        + emission_columns
    ].head()
)

print("Generation units assumed to be daily averages:",
      GENERATION_COLUMNS_ARE_DAILY_AVERAGES)
print("Separate imports generation column available:",
      "imports" in daily_data.columns)

### 3.3 Check generation totals and emissions limitations

The seven available generation categories are compared with `net_generation`. The dataset has no separate imports generation column, so imported electricity emissions cannot be calculated independently. The emissions result is therefore an estimate based on the generation categories available in this dataset.

In [ ]:
daily_data["generation_sum"] = daily_data[generation_fuels].sum(axis=1)
daily_data["generation_difference"] = (
    daily_data["net_generation"] - daily_data["generation_sum"]
)

print(
    "Maximum absolute difference between net generation and listed sources:",
    daily_data["generation_difference"].abs().max()
)

display(
    daily_data[
        ["date", "net_generation", "generation_sum", "generation_difference"]
    ].head()
)

plt.figure(figsize=(12, 4))
plt.plot(daily_data["date"], daily_data["daily_co2_kg"], linewidth=0.8)
plt.title("Estimated Daily Grid CO₂ Emissions")
plt.xlabel("Date")
plt.ylabel("Estimated CO₂ (kg)")
plt.grid(True, alpha=0.3)
plt.show()

### 3.4 Create calendar variables and the modeling dataset

The candidate models use `day_of_week` and `day_of_year` as calendar variables. The month and year columns are retained for inspection but are not added to the candidate feature sets, so the model comparisons change one planned feature at a time.

In [ ]:
daily_data["day_of_week"] = daily_data["date"].dt.dayofweek
daily_data["day_of_year"] = daily_data["date"].dt.dayofyear
daily_data["month"] = daily_data["date"].dt.month
daily_data["year"] = daily_data["date"].dt.year

model_columns = [
    "date", "temp_f", "demand_mwh", "daily_co2_kg",
    "day_of_week", "day_of_year", "month", "year"
]

model_data = (
    daily_data[model_columns]
    .dropna()
    .sort_values("date")
    .reset_index(drop=True)
    .copy()
)

print("Modeling observations:", len(model_data))
print("Modeling date range:",
      model_data["date"].min(), "to", model_data["date"].max())
display(model_data.head())

## 4. Cold-versus-Warm Experiments

### 4.1 Experiment design

Two outcome comparisons test the two predictions:

- **Experiment A:** daily average electricity demand for cold days versus warm days.
- **Experiment B:** estimated daily grid CO₂ emissions for cold days versus warm days.

**Held fixed:** the same prepared daily dataset, date range, temperature thresholds, group definitions, and comparison calculations.

**Varied:** the outcome being measured (`demand_mwh` or `daily_co2_kg`) and, within each comparison, the temperature group (cold versus warm).

The cold group is at or below the 25th percentile of `temp_f`; the warm group is at or above the 75th percentile. The middle half is excluded. Ties at the thresholds may make group sizes slightly different from exactly 25%.

In [ ]:
cold_threshold = model_data["temp_f"].quantile(0.25)
warm_threshold = model_data["temp_f"].quantile(0.75)

model_data["temperature_group"] = np.select(
    [
        model_data["temp_f"] <= cold_threshold,
        model_data["temp_f"] >= warm_threshold
    ],
    ["Cold", "Warm"],
    default="Middle"
)

print(f"Cold threshold (25th percentile): {cold_threshold:.2f} °F")
print(f"Warm threshold (75th percentile): {warm_threshold:.2f} °F")
display(model_data["temperature_group"].value_counts().rename("days").to_frame())

### 4.2 Compare group means and medians

The absolute difference is `cold mean − warm mean`. The percentage difference is `(cold mean − warm mean) / warm mean × 100`. A positive difference means the cold-group average is higher.

In [ ]:
def compare_temperature_groups(data, outcome):
    cold = data.loc[data["temperature_group"] == "Cold", outcome].dropna()
    warm = data.loc[data["temperature_group"] == "Warm", outcome].dropna()

    cold_mean = cold.mean()
    warm_mean = warm.mean()
    difference = cold_mean - warm_mean
    percent_difference = (
        difference / warm_mean * 100 if warm_mean != 0 else np.nan
    )

    return {
        "outcome": outcome,
        "cold_n": len(cold),
        "warm_n": len(warm),
        "cold_mean": cold_mean,
        "warm_mean": warm_mean,
        "cold_median": cold.median(),
        "warm_median": warm.median(),
        "cold_minus_warm": difference,
        "percent_difference": percent_difference
    }

experiment_results = pd.DataFrame([
    compare_temperature_groups(model_data, "demand_mwh"),
    compare_temperature_groups(model_data, "daily_co2_kg")
])

display(experiment_results)

### 4.3 Visualize the two comparisons

The scatter plots show the relationship across all daily observations. The boxplots show the distributions in the cold and warm groups used in the experiment.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

axes[0].scatter(model_data["temp_f"], model_data["demand_mwh"], alpha=0.45)
axes[0].set_title("Temperature vs. Daily Average Electricity Demand")
axes[0].set_xlabel("Daily average temperature (°F)")
axes[0].set_ylabel("Daily average demand (MWh)")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(model_data["temp_f"], model_data["daily_co2_kg"], alpha=0.45)
axes[1].set_title("Temperature vs. Estimated Daily Grid CO₂")
axes[1].set_xlabel("Daily average temperature (°F)")
axes[1].set_ylabel("Estimated daily CO₂ (kg)")
axes[1].grid(True, alpha=0.3)

plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

demand_groups = [
    model_data.loc[model_data["temperature_group"] == group, "demand_mwh"].dropna()
    for group in ["Cold", "Warm"]
]
axes[0].boxplot(demand_groups, tick_labels=["Cold", "Warm"])
axes[0].set_title("Daily Average Demand: Cold vs. Warm")
axes[0].set_ylabel("Demand (MWh)")
axes[0].grid(True, axis="y", alpha=0.3)

emission_groups = [
    model_data.loc[model_data["temperature_group"] == group, "daily_co2_kg"].dropna()
    for group in ["Cold", "Warm"]
]
axes[1].boxplot(emission_groups, tick_labels=["Cold", "Warm"])
axes[1].set_title("Estimated Grid CO₂: Cold vs. Warm")
axes[1].set_ylabel("Estimated daily CO₂ (kg)")
axes[1].grid(True, axis="y", alpha=0.3)

plt.show()

### 4.6 Compare Cold and Warm Temperature Groups

Two comparisons will be performed using the temperature groups.

**Experiment 1: Daily Average Electricity Consumption**

The mean and median daily average hourly electricity consumption will be compared between the coldest and warmest temperature groups.

**Experiment 2: Daily Estimated Grid CO₂ Emissions**

The mean and median estimated daily grid CO₂ emissions will be compared between the coldest and warmest temperature groups.

For each experiment, the following results will be reported:

- Number of observations in each group.
- Cold-group and warm-group means.
- Cold-group and warm-group medians.
- Absolute difference.
- Percentage difference.

The results will be used to evaluate whether the observed differences support the two predictions in Hypothesis 2.

In [ ]:
def compare_temperature_groups(data, outcome):
    cold = data.loc[data["temperature_group"] == "Cold", outcome].dropna()
    warm = data.loc[data["temperature_group"] == "Warm", outcome].dropna()

    cold_mean = cold.mean()
    warm_mean = warm.mean()
    difference = cold_mean - warm_mean
    percent_difference = (
        difference / warm_mean * 100 if warm_mean != 0 else np.nan
    )

    return {
        "outcome": outcome,
        "cold_n": len(cold),
        "warm_n": len(warm),
        "cold_mean": cold_mean,
        "warm_mean": warm_mean,
        "cold_median": cold.median(),
        "warm_median": warm.median(),
        "cold_minus_warm": difference,
        "percent_difference": percent_difference
    }

experiment_results = pd.DataFrame([
    compare_temperature_groups(model_data, "demand_mwh"),
    compare_temperature_groups(model_data, "daily_co2_kg")
])

display(experiment_results)

## 5. R7 — Candidate Models

### 5.1 Model-construction question

The model experiment asks whether daily average temperature adds predictive information beyond calendar variables when predicting:

1. `demand_mwh`
2. `daily_co2_kg`

All candidates use the same **Linear Regression** algorithm and the same `StandardScaler` + regression pipeline. The candidate variable sets change in a planned sequence.

### 5.2 Candidate models

| Candidate model | Variables | Reason for inclusion |
|---|---|---|
| **Calendar baseline** | `day_of_week`, `day_of_year` | Establishes performance using calendar patterns without temperature. |
| **Calendar + temperature** | Calendar variables + `temp_f_centered` | Tests whether temperature adds predictive information beyond calendar patterns. |
| **Calendar + temperature + squared temperature** | Calendar variables + `temp_f_centered` + `temp_f_squared` | Tests whether a squared temperature term captures a nonlinear pattern. |

The squared term allows a curved temperature relationship. Each candidate adds one feature to the previous feature set, while the algorithm and preprocessing remain unchanged.

### 5.3 Fixed temperature transformation

Temperature is centered at a fixed reference of 65°F. This is a constant, not a statistic estimated from the validation or testing data. It avoids using information from later data when creating the squared term.

The feature names below are descriptive Python identifiers; the lecture terminology is used in headings, model labels, and result displays.

In [ ]:
TEMP_REFERENCE_F = 65.0

model_data["temp_f_centered"] = model_data["temp_f"] - TEMP_REFERENCE_F
model_data["temp_f_squared"] = model_data["temp_f_centered"] ** 2

feature_sets = {
    "calendar_baseline": [
        "day_of_week",
        "day_of_year"
    ],
    "calendar_temperature": [
        "day_of_week",
        "day_of_year",
        "temp_f_centered"
    ],
    "calendar_temperature_squared": [
        "day_of_week",
        "day_of_year",
        "temp_f_centered",
        "temp_f_squared"
    ]
}

candidate_model_labels = {
    "calendar_baseline": "Calendar baseline",
    "calendar_temperature": "Calendar + temperature",
    "calendar_temperature_squared": "Calendar + temperature + squared temperature"
}

target_variables = {
    "electricity_demand": "demand_mwh",
    "grid_co2_emissions": "daily_co2_kg"
}

for model_key, features in feature_sets.items():
    print(candidate_model_labels[model_key], "->", features)

print("\nPrediction targets:", target_variables)

### 5.4 Prediction Targets

Each candidate feature set will be evaluated separately against two prediction targets.

- `demand_mwh`: Daily average electricity demand in megawatt-hours.
- `daily_co2_kg`: Estimated daily grid CO2 emissions in kilograms.

This produces six candidate model configurations: three feature sets for each of the two prediction targets.

The primary comparison will be validation RMSE. MAE and R-squared will also be recorded to provide additional information about prediction error and model fit.

In [ ]:
# Define the two prediction targets
target_variables = {
    "electricity_demand": "demand_mwh",
    "grid_co2_emissions": "daily_co2_kg"
}

print("Prediction targets:")
for name, column in target_variables.items():
    print(f"{name}: {column}")

## 6. Shared Chronological Split and Validation Design

### 6.1 One split reused by every candidate

Because the observations are daily and ordered in time, a chronological split is more appropriate than a random split. It simulates predicting later dates from earlier observations and prevents future dates from entering model construction.

The dataset is divided once:

- **Training data:** first 70%; used to fit candidate models.
- **Validation data:** next 15%; used to compare candidates and select a model.
- **Testing data:** final 15%; held back until final testing.

The exact same observations are used for all six candidate runs (three candidate models × two targets).

In [ ]:
model_data = model_data.sort_values("date").reset_index(drop=True).copy()

total_rows = len(model_data)
training_end = int(total_rows * 0.70)
validation_end = int(total_rows * 0.85)

training_data = model_data.iloc[:training_end].copy()
validation_data = model_data.iloc[training_end:validation_end].copy()
testing_data = model_data.iloc[validation_end:].copy()

split_summary = pd.DataFrame([
    {
        "subset": "Training data",
        "rows": len(training_data),
        "first_date": training_data["date"].min(),
        "last_date": training_data["date"].max()
    },
    {
        "subset": "Validation data",
        "rows": len(validation_data),
        "first_date": validation_data["date"].min(),
        "last_date": validation_data["date"].max()
    },
    {
        "subset": "Testing data",
        "rows": len(testing_data),
        "first_date": testing_data["date"].min(),
        "last_date": testing_data["date"].max()
    }
])

display(split_summary)

assert training_data["date"].max() < validation_data["date"].min()
assert validation_data["date"].max() < testing_data["date"].min()
assert len(training_data) + len(validation_data) + len(testing_data) == len(model_data)

print("Chronological split verified. No dates overlap.")

### 6.2 Time-series cross-validation within training data

Three expanding-window folds are created from **training data only**. Each fold trains on earlier observations and validates on the next chronological block. The validation subset remains the common candidate-selection dataset; the testing subset remains untouched.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

time_series_cv = TimeSeriesSplit(n_splits=3)

fold_rows = []
for fold_number, (fold_training_indices, fold_validation_indices) in enumerate(
    time_series_cv.split(training_data), start=1
):
    fold_training = training_data.iloc[fold_training_indices]
    fold_validation = training_data.iloc[fold_validation_indices]

    fold_rows.append({
        "fold": fold_number,
        "training_rows": len(fold_training),
        "training_start": fold_training["date"].min(),
        "training_end": fold_training["date"].max(),
        "validation_rows": len(fold_validation),
        "validation_start": fold_validation["date"].min(),
        "validation_end": fold_validation["date"].max()
    })

display(pd.DataFrame(fold_rows))

### 6.3 Evaluation metrics and interpretation

- **MAE:** average absolute prediction error; lower is better.
- **RMSE:** prediction error that penalizes larger errors more heavily; lower is better.
- **R²:** proportion of target variation explained by the model; values closer to 1 generally indicate a stronger fit, while negative values are possible.

The mean and standard deviation of cross-validation scores describe average performance and variation across chronological folds. The **training–validation RMSE gap** is validation RMSE minus training RMSE. A larger positive gap can indicate a generalization problem, but it is not a direct mathematical measurement of model bias or variance.

Lecture 5.2 describes high bias as systematic error associated with underfitting, and high variance as predictions changing substantially across training samples. The cross-validation spread is used here as evidence about stability, not as a formal estimate of variance.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

def calculate_metrics(actual, predicted):
    return {
        "mae": mean_absolute_error(actual, predicted),
        "rmse": np.sqrt(mean_squared_error(actual, predicted)),
        "r2": r2_score(actual, predicted)
    }

def build_regression_model():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("regression", LinearRegression())
    ])

## 7. R7 — Model Construction and MLflow Experiment Tracking

### 7.1 Connect to the shared MLflow tracking server

Use environment variables for credentials where possible. If they are not set, the notebook prompts for them without displaying the password. Do not commit real credentials to the repository.

AI Usage: Next cell, the code is to help me run this notebook locally, instead of the VM.

In [ ]:
# Connect to the provided remote MLflow server

import os
import requests
import mlflow
import mlflow.sklearn

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Configure remote tracking
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_SERVER
os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_SERVER)

# Confirm server access
reply = requests.get(
    MLFLOW_SERVER.rstrip("/") + "/api/2.0/mlflow/experiments/search",
    params={"max_results": 1},
    auth=(MLFLOW_USERNAME, MLFLOW_PASSWORD),
    timeout=10
)

print("Server response:", reply.status_code)

if reply.status_code == 200:
    EXPERIMENT = f"Group1-Climate-Energy-{MLFLOW_USERNAME}"
    mlflow.set_experiment(EXPERIMENT)

    print("Connected successfully!")
    print("Tracking URI:", mlflow.get_tracking_uri())
    print("Experiment:", EXPERIMENT)

elif reply.status_code == 401:
    raise Exception("Incorrect MLflow username or password.")

elif reply.status_code == 403:
    raise Exception("Your account does not have permission to access MLflow.")

else:
    raise Exception(f"MLflow server returned HTTP {reply.status_code}")

In [ ]:
candidate_models = {}
candidate_model_results = []
candidate_run_ids = {}

for target_name, target_column in target_variables.items():
    for feature_name, features in feature_sets.items():

        run_name = f"H2_{target_name}_{feature_name}"

        X_training = training_data[features]
        y_training = training_data[target_column]
        X_validation = validation_data[features]
        y_validation = validation_data[target_column]

        model = build_regression_model()
        model.fit(X_training, y_training)

        training_metrics = calculate_metrics(
            y_training, model.predict(X_training)
        )
        validation_metrics = calculate_metrics(
            y_validation, model.predict(X_validation)
        )

        fold_metric_values = {"mae": [], "rmse": [], "r2": []}
        fold_rmse_values = []

        for fold_number, (fold_train_idx, fold_validation_idx) in enumerate(
            time_series_cv.split(training_data), start=1
        ):
            fold_training = training_data.iloc[fold_train_idx]
            fold_validation = training_data.iloc[fold_validation_idx]

            fold_model = build_regression_model()
            fold_model.fit(
                fold_training[features],
                fold_training[target_column]
            )

            fold_predictions = fold_model.predict(
                fold_validation[features]
            )
            fold_metrics = calculate_metrics(
                fold_validation[target_column],
                fold_predictions
            )

            for metric_name in fold_metric_values:
                fold_metric_values[metric_name].append(
                    fold_metrics[metric_name]
                )
            fold_rmse_values.append(fold_metrics["rmse"])

        cv_summary = {}
        for metric_name, values in fold_metric_values.items():
            cv_summary[f"cv_{metric_name}_mean"] = float(np.mean(values))
            cv_summary[f"cv_{metric_name}_std"] = float(np.std(values, ddof=1))

        training_validation_rmse_gap = (
            validation_metrics["rmse"] - training_metrics["rmse"]
        )

        with mlflow.start_run(run_name=run_name) as active_run:
            mlflow.set_tag("model_stage", "candidate")
            mlflow.set_tag("hypothesis", "H2")
            mlflow.set_tag("experiment_group", MLFLOW_EXPERIMENT)
            mlflow.set_tag("candidate_label", candidate_model_labels[feature_name])

            mlflow.log_params({
                "hypothesis": "H2",
                "algorithm": "LinearRegression",
                "target": target_name,
                "target_column": target_column,
                "candidate_model": feature_name,
                "candidate_label": candidate_model_labels[feature_name],
                "features": ", ".join(features),
                "preprocessing": "StandardScaler",
                "data_split": "chronological_70_15_15",
                "cross_validation": "TimeSeriesSplit",
                "cv_folds": 3,
                "temperature_reference_f": TEMP_REFERENCE_F
            })

            for metric_name, value in training_metrics.items():
                mlflow.log_metric(f"training_{metric_name}", float(value))
            for metric_name, value in validation_metrics.items():
                mlflow.log_metric(f"validation_{metric_name}", float(value))
            for metric_name, value in cv_summary.items():
                mlflow.log_metric(metric_name, float(value))

            mlflow.log_metric(
                "training_validation_rmse_gap",
                float(training_validation_rmse_gap)
            )

            for fold_number, fold_rmse in enumerate(fold_rmse_values, start=1):
                mlflow.log_metric(f"cv_fold_{fold_number}_rmse", float(fold_rmse))

            mlflow.log_dict({
                "hypothesis": "H2",
                "target": target_column,
                "candidate_model": feature_name,
                "candidate_label": candidate_model_labels[feature_name],
                "features": features,
                "algorithm": "LinearRegression",
                "preprocessing": "StandardScaler",
                "split": "chronological 70/15/15",
                "cross_validation": "3-fold expanding-window TimeSeriesSplit",
                "temperature_reference_f": TEMP_REFERENCE_F
            }, "experiment_configuration.json")

            mlflow.sklearn.log_model(
                sk_model=model,
                artifact_path="model"
            )

            run_id = active_run.info.run_id

        candidate_models[(target_name, feature_name)] = model
        candidate_run_ids[(target_name, feature_name)] = run_id

        candidate_model_results.append({
            "run_name": run_name,
            "run_id": run_id,
            "target": target_name,
            "candidate_model": feature_name,
            "candidate_label": candidate_model_labels[feature_name],
            "features": ", ".join(features),
            "training_mae": training_metrics["mae"],
            "training_rmse": training_metrics["rmse"],
            "training_r2": training_metrics["r2"],
            "validation_mae": validation_metrics["mae"],
            "validation_rmse": validation_metrics["rmse"],
            "validation_r2": validation_metrics["r2"],
            "training_validation_rmse_gap": training_validation_rmse_gap,
            **cv_summary
        })

candidate_model_comparison = pd.DataFrame(candidate_model_results)

print("Candidate runs completed:", len(candidate_model_comparison))
display(candidate_model_comparison.sort_values(
    ["target", "validation_rmse"]
))

### 7.3 Compare candidate model performance

Validation RMSE is the primary selection metric. Cross-validation mean and standard deviation, validation MAE and R², and the training–validation gap provide additional evidence. A more complex candidate is not automatically better: its improvement must generalize to validation data and remain reasonably stable across folds.

In [ ]:
for target_name in target_variables:
    target_results = candidate_model_comparison[
        candidate_model_comparison["target"] == target_name
    ].sort_values("validation_rmse")

    plt.figure(figsize=(9, 5))
    plt.bar(
        target_results["candidate_label"],
        target_results["validation_rmse"]
    )
    plt.title(f"Candidate Model Comparison — {target_name}")
    plt.ylabel("Validation RMSE (lower is better)")
    plt.xticks(rotation=20, ha="right")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

# Cross-validation RMSE: mean with one standard deviation
for target_name in target_variables:
    target_results = candidate_model_comparison[
        candidate_model_comparison["target"] == target_name
    ].copy()

    x = np.arange(len(target_results))
    plt.figure(figsize=(9, 5))
    plt.errorbar(
        x,
        target_results["cv_rmse_mean"],
        yerr=target_results["cv_rmse_std"],
        fmt="o",
        capsize=5
    )
    plt.xticks(x, target_results["candidate_label"], rotation=20, ha="right")
    plt.title(f"Cross-Validation RMSE Stability — {target_name}")
    plt.ylabel("Cross-validation RMSE (mean ± standard deviation)")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

### 7.4 Select one candidate model per prediction target

For each target, select the candidate with the lowest validation RMSE. Review cross-validation spread and the training–validation gap before accepting the selection. The selection is made without looking at testing-data scores.

In [ ]:
selected_candidate_rows = (
    candidate_model_comparison
    .sort_values(["target", "validation_rmse"])
    .groupby("target", as_index=False)
    .first()
)

selected_models = {}
for _, row in selected_candidate_rows.iterrows():
    target_name = row["target"]
    feature_name = row["candidate_model"]
    selected_models[target_name] = {
        "candidate_model": feature_name,
        "candidate_label": candidate_model_labels[feature_name],
        "features": feature_sets[feature_name],
        "model": candidate_models[(target_name, feature_name)],
        "candidate_run_id": row["run_id"]
    }

display(selected_candidate_rows[
    [
        "target", "candidate_label",
        "validation_rmse", "cv_rmse_mean", "cv_rmse_std",
        "training_validation_rmse_gap"
    ]
])

### 7.5 Retrieve and export the MLflow run table

The exported run table includes the MLflow run name, run ID, all logged parameters, and all logged metrics for the candidate runs. This provides a repository copy of the experiment record in addition to the remote MLflow artifacts.

In [ ]:
mlflow_experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
if mlflow_experiment is None:
    raise ValueError(
        f"MLflow experiment '{MLFLOW_EXPERIMENT}' was not found. "
        "Check the tracking URI and experiment name."
    )

all_experiment_runs = mlflow.search_runs(
    experiment_ids=[mlflow_experiment.experiment_id]
)

candidate_mlflow_runs = all_experiment_runs[
    all_experiment_runs["tags.model_stage"] == "candidate"
].copy()

candidate_mlflow_runs = candidate_mlflow_runs.rename(
    columns={"tags.mlflow.runName": "run_name"}
)

# Keep run identity, all parameter columns, all metric columns, and useful tags.
identity_columns = [
    column for column in [
        "run_name", "run_id", "status", "start_time", "end_time"
    ] if column in candidate_mlflow_runs.columns
]
parameter_columns = sorted(
    column for column in candidate_mlflow_runs.columns
    if column.startswith("params.")
)
metric_columns = sorted(
    column for column in candidate_mlflow_runs.columns
    if column.startswith("metrics.")
)
tag_columns = sorted(
    column for column in candidate_mlflow_runs.columns
    if column.startswith("tags.")
)

mlflow_run_table = candidate_mlflow_runs[
    identity_columns + parameter_columns + metric_columns + tag_columns
].copy()

mlflow_run_table.to_csv("hypothesis_2_mlflow_runs.csv", index=False)

print("Candidate MLflow runs exported:", len(mlflow_run_table))
display(mlflow_run_table)

## 8. R8 — Model Selection, Validation, and Testing

### 8.1 Validation evidence for the selected models

For each selected candidate, report:
- validation MAE, RMSE, and R²;
- cross-validation RMSE mean and standard deviation;
- training–validation RMSE gap.

The validation data was not used to fit the candidate model. The cross-validation folds use training data only and provide evidence about performance stability across earlier time periods.

In [ ]:
validation_evidence = selected_candidate_rows[
    [
        "target", "candidate_label",
        "training_rmse", "validation_rmse",
        "validation_mae", "validation_r2",
        "cv_rmse_mean", "cv_rmse_std",
        "training_validation_rmse_gap"
    ]
].copy()

display(validation_evidence)

### 8.2 Final model construction and one-time testing

After model selection, each selected model is refitted using the combined training and validation data. The reserved testing data is then scored once for each target. No feature selection or tuning is performed using the testing results.

In [ ]:
development_data = pd.concat(
    [training_data, validation_data],
    ignore_index=True
)

final_models = {}
final_test_rows = []
test_predictions_by_target = {}

for target_name, selection in selected_models.items():
    target_column = target_variables[target_name]
    features = selection["features"]

    final_model = build_regression_model()
    final_model.fit(
        development_data[features],
        development_data[target_column]
    )

    # The reserved testing subset is evaluated once here.
    predictions = final_model.predict(testing_data[features])
    test_metrics = calculate_metrics(
        testing_data[target_column],
        predictions
    )

    final_models[target_name] = final_model
    test_predictions_by_target[target_name] = predictions

    final_test_rows.append({
        "target": target_name,
        "selected_candidate_model": selection["candidate_model"],
        "selected_candidate_label": selection["candidate_label"],
        "test_mae": test_metrics["mae"],
        "test_rmse": test_metrics["rmse"],
        "test_r2": test_metrics["r2"]
    })

    with mlflow.start_run(
        run_name=f"FINAL_TEST_{target_name}_{selection['candidate_model']}"
    ) as final_run:
        mlflow.set_tag("model_stage", "final_test")
        mlflow.set_tag("hypothesis", "H2")
        mlflow.log_params({
            "algorithm": "LinearRegression",
            "target": target_name,
            "selected_candidate_model": selection["candidate_model"],
            "features": ", ".join(features),
            "training_data": "training_plus_validation",
            "testing_data": "reserved_chronological_final_15_percent"
        })
        for metric_name, value in test_metrics.items():
            mlflow.log_metric(f"test_{metric_name}", float(value))
        mlflow.sklearn.log_model(
            sk_model=final_model,
            artifact_path="final_model"
        )

final_test_results = pd.DataFrame(final_test_rows)
display(final_test_results)

### 8.3 Visualize final testing performance

The first chart compares actual and predicted values on the reserved testing dates. The second chart shows residuals (actual minus predicted). These visuals help identify systematic errors or unusually large prediction errors.

In [ ]:
for target_name, target_column in target_variables.items():
    actual = testing_data[target_column].to_numpy()
    predicted = test_predictions_by_target[target_name]
    residuals = actual - predicted

    lower = min(actual.min(), predicted.min())
    upper = max(actual.max(), predicted.max())

    plt.figure(figsize=(6, 6))
    plt.scatter(actual, predicted, alpha=0.65)
    plt.plot([lower, upper], [lower, upper], linestyle="--")
    plt.title(f"Actual vs. Predicted — {target_name} (Testing Data)")
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.scatter(predicted, residuals, alpha=0.65)
    plt.axhline(0, linestyle="--")
    plt.title(f"Testing Residuals — {target_name}")
    plt.xlabel("Predicted value")
    plt.ylabel("Residual (actual − predicted)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### 8.4 Compare validation and testing scores

A higher testing RMSE than validation RMSE may indicate that the selected model generalizes less well to the later period. This comparison is descriptive; the testing scores do not trigger another round of model selection.

In [ ]:
validation_test_comparison = selected_candidate_rows[
    ["target", "candidate_label", "validation_rmse"]
].merge(
    final_test_results[["target", "test_mae", "test_rmse", "test_r2"]],
    on="target",
    how="left"
)

validation_test_comparison["test_minus_validation_rmse"] = (
    validation_test_comparison["test_rmse"]
    - validation_test_comparison["validation_rmse"]
)

display(validation_test_comparison)

### 8.5 Hypothesis Verdicts

The cold-versus-warm experiments and bootstrap intervals determine the hypothesis verdict. Model scores provide additional predictive context, but predictive performance alone does not prove that temperature caused changes in demand or emissions.

For each prediction:
- **Supported:** the cold-minus-warm 95% interval is entirely above zero.
- **Rejected:** the interval is entirely below zero.
- **Unresolved:** the interval includes zero.

The original hourly hypothesis was **revised** to a daily hypothesis because of the temporal resolution of the prepared data. That revision is documented in Section 1; the evidence verdict below applies to the revised daily hypothesis.

In [ ]:
def prediction_verdict(lower, upper):
    if lower > 0:
        return "Supported"
    if upper < 0:
        return "Rejected"
    return "Unresolved"

verdict_rows = []
for _, row in bootstrap_results.iterrows():
    verdict_rows.append({
        "prediction": row["outcome"],
        "cold_minus_warm": row["cold_minus_warm"],
        "ci_95_lower": row["ci_95_lower"],
        "ci_95_upper": row["ci_95_upper"],
        "verdict": prediction_verdict(
            row["ci_95_lower"], row["ci_95_upper"]
        )
    })

prediction_verdicts = pd.DataFrame(verdict_rows)
display(prediction_verdicts)

verdict_set = set(prediction_verdicts["verdict"])
if verdict_set == {"Supported"}:
    overall_hypothesis_verdict = "Supported"
elif verdict_set == {"Rejected"}:
    overall_hypothesis_verdict = "Rejected"
else:
    overall_hypothesis_verdict = "Unresolved"

print("Overall verdict for revised daily Hypothesis 2:",
      overall_hypothesis_verdict)

### 8.6 Export the M2 Evidence

The following files preserve the numerical evidence required for the repository:
- `hypothesis_2_mlflow_runs.csv`: MLflow run name, run ID, parameters, metrics, and tags.
- `hypothesis_2_cold_warm_results.csv`: cold-versus-warm means, medians, counts, and differences.
- `hypothesis_2_candidate_model_comparison.csv`: candidate model validation and cross-validation results.
- `hypothesis_2_final_test_results.csv`: one-time final testing results.
- `hypothesis_2_validation_test_comparison.csv`: validation versus testing scores.
- `hypothesis_2_verdicts.csv`: prediction-level and overall verdict evidence.

In [ ]:
mlflow_run_table.to_csv("hypothesis_2_mlflow_runs.csv", index=False)
experiment_results.to_csv(
    "hypothesis_2_cold_warm_results.csv", index=False
)

candidate_model_comparison.to_csv(
    "hypothesis_2_candidate_model_comparison.csv", index=False
)
final_test_results.to_csv(
    "hypothesis_2_final_test_results.csv", index=False
)
validation_test_comparison.to_csv(
    "hypothesis_2_validation_test_comparison.csv", index=False
)
prediction_verdicts.to_csv(
    "hypothesis_2_verdicts.csv", index=False
)

print("M2 evidence files exported.")

### 8.7 Limitations and Conclusion

**Limitations**

1. **Geographic coverage:** The grid measurements represent the broader MCPA balancing authority, not only the city or the 10,000 EVs.
2. **Daily aggregation:** Daily averages cannot identify hourly charging behavior, nighttime charging, or short-lived demand changes.
3. **Mixed demand sources:** The demand measurement includes residential, commercial, industrial, streetlight, and EV load.
4. **Estimated emissions:** The estimate depends on the available generation categories and fuel factors. There is no separate imports generation column, so imports emissions cannot be calculated independently.
5. **Generation units:** The emissions calculation depends on whether source generation values are average hourly values or daily totals. The configuration in Section 3 must match the source documentation.
6. **Other influences:** Seasonality, day of week, heating fuel, economic activity, and generation availability can influence demand and emissions.
7. **Limited time period:** The dataset covers 2022 and 2023 only.
8. **Model limitations:** Linear Regression may not capture every nonlinear relationship. The candidate set is intentionally limited so that the model comparisons remain controlled and interpretable.
9. **Uncertainty:** The independent bootstrap does not preserve serial dependence between nearby days; its confidence intervals are approximate.

### Conclusion template

Complete this section after running all cells. Use the actual numeric output; do not insert assumed results.

- **Electricity-demand comparison:** cold mean = [value], warm mean = [value], cold-minus-warm = [value], 95% interval = [lower, upper], verdict = [Supported / Rejected / Unresolved].
- **Grid-emissions comparison:** cold mean = [value], warm mean = [value], cold-minus-warm = [value], 95% interval = [lower, upper], verdict = [Supported / Rejected / Unresolved].
- **Selected demand model:** [candidate label], validation RMSE = [value], CV RMSE mean ± standard deviation = [values], training–validation RMSE gap = [value], testing RMSE = [value].
- **Selected emissions model:** [candidate label], validation RMSE = [value], CV RMSE mean ± standard deviation = [values], training–validation RMSE gap = [value], testing RMSE = [value].
- **Overall verdict for revised daily Hypothesis 2:** [Supported / Rejected / Unresolved].

Interpret the findings as associations and predictive patterns within the available balancing-area data, not as proof that cold weather caused the observed changes or that EV adoption caused all observed demand and emissions.

## 11. Limitations and Conclusion

### 11.1 Study Limitations

Several limitations should be considered when interpreting the results.

**Geographic Coverage:** The electricity grid data represents the broader MCPA balancing authority rather than only the specific city where the 10,000 EVs were introduced. Therefore, the analysis cannot attribute all observed changes in electricity demand or emissions directly to the EVs.

**Daily Aggregation:** The analysis uses daily averages rather than hourly observations. This limits the ability to identify specific charging periods, nighttime demand patterns, and short-term changes in electricity consumption.

**Multiple Sources of Electricity Demand:** The electricity demand data includes homes, businesses, industry, streetlights, and EV chargers. The individual contribution of EV charging cannot be separated using the available data.

**Estimated Grid Emissions:** Estimated CO₂ emissions depend on the available generation data and assigned fuel emissions factors. The combined dataset does not provide a separate imports generation column, limiting the ability to calculate imported electricity emissions independently.

**Seasonal and External Influences:** Temperature is not the only factor affecting electricity demand and emissions. Day of the week, seasonal changes, heating requirements, economic activity, and generation availability may also influence the results.

**Limited Observation Period:** The dataset covers 2022 and 2023. The findings may not represent longer-term electricity consumption or generation patterns.

**Predictive Model Limitations:** Linear Regression may not capture every complex relationship between temperature, electricity demand, and grid emissions. Model performance is also limited by the available features and observation period.

Because of these limitations, the findings should be interpreted as observed associations and predictive relationships rather than proof of direct causation.

### 11.2 Hypothesis 2 Conclusion

This experiment examined whether lower daily average outdoor temperatures were associated with higher daily average electricity demand and higher estimated daily grid CO₂ emissions.

The coldest and warmest temperature groups were compared using daily observations from 2022 and 2023. Three candidate feature sets were evaluated using Linear Regression to determine whether temperature and temperature-squared features improved predictions beyond calendar information.

Model selection was based on validation performance and chronological cross-validation. The selected models were then evaluated on the reserved test dataset.

**Final Findings:**

- **Electricity Demand:** [Insert cold-versus-warm difference, confidence interval, and selected model results.]
- **Estimated Grid CO₂ Emissions:** [Insert cold-versus-warm difference, confidence interval, and selected model results.]
- **Best Electricity Demand Model:** [Insert selected feature set.]
- **Best Grid CO₂ Emissions Model:** [Insert selected feature set.]
- **Final Test Performance:** [Insert test RMSE and R² for both outcomes.]
- **Overall Hypothesis Verdict:** [Insert supported, partially supported, rejected, or unresolved.]

The findings will indicate whether colder daily temperatures were associated with higher electricity demand and estimated grid CO₂ emissions during the study period.

The conclusions will remain limited to the available balancing-area data and the two-year observation period.